# Gauntlet day — start here

Seven submissions want capital. You have their **records** — trade blotters, registry
declarations, claimed stats — and the **market** they claim to have traded. No source code.
Your build target is `report.py`: one pipeline, run unchanged over all seven, returning
**PASS / SUSPECT (with conditions) / FAIL** with evidence behind every line.

Read `../docs/submissions-book.pdf` and `../docs/six-sections-handout.pdf` first.
This notebook just proves the records hold together and gets everything loaded.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import sys; sys.path.insert(0, "../registry")
from repricer import load_market, load_blotter, reprice_mid, daily_pnl, sharpe
from loader import load_registry

market = load_market("../data")   # ~30s: the book table is 524k rows
registry = load_registry("../registry")
sorted(registry)

## The market
Three levels per side, snapshotted every 15 minutes for the last 10 hours before gate.
There is no mid column — mid is *derived*, `(bid_px_1 + ask_px_1)/2`. Nothing in these
files says anyone fills you there.

In [ ]:
market["book"].head(3)

## A registry entry
Everything except `claimed` is the submitter's own declaration — params, trial count,
dev window, data dependencies. **Unverified.** Checking declarations against records is
your job (Lineage), and computing on the declared numbers is also your job (Luck deflates
Sharpe by *declared* trials). The empty `validation:` block is what your pipeline fills.

In [ ]:
registry["windfall"]

## A blotter
The behavioural record: every trade the strategy claims it did. Same schema for all
seven. Every position is opened and closed through trades, so product P&L is just
sells minus buys — at whatever price column you choose to value the trades.

In [ ]:
blot = load_blotter("../blotters/windfall-blotter.csv")
blot.head()

## Consistency: reproduce the book from the records
Reprice every trade at the book mid and rebuild daily P&L over the declared backtest
window. It should match the claimed numbers to rounding — for **all seven**, including
the broken ones. That's the point: a leaked, overfit, or friction-doomed strategy is
perfectly self-consistent at mid. Consistency is where validation starts, not where
it ends.

In [ ]:
rows = []
curves = {}
for key, entry in registry.items():
    b = reprice_mid(load_blotter(f"../blotters/{key}-blotter.csv"), market)
    w = (str(entry["backtest_window"]["start"]), str(entry["backtest_window"]["end"]))
    d = daily_pnl(b, "mid_price", window=w)
    curves[key] = d.cumsum()
    rows.append({"submission": key,
                 "claimed_pnl": entry["claimed"]["total_pnl_eur"],
                 "repriced_pnl": round(d.sum()),
                 "claimed_sharpe": entry["claimed"]["sharpe_ann"],
                 "rebuilt_sharpe": round(sharpe(d), 2),
                 "days": entry["backtest_window"]["days"],
                 "trades": len(b)})
league = pd.DataFrame(rows).set_index("submission") \
    .sort_values("rebuilt_sharpe", ascending=False)
league

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for key, eq in curves.items():
    ax.plot(eq.index, eq.values, lw=1.4, label=key)
ax.legend(ncol=4, fontsize=8); ax.set_ylabel("cum P&L at mid (EUR)")
ax.set_title("the naive league table — every one of these charts is 'true'")
plt.tight_layout()

## The deliverable's shape — a worked example on a FAKE submission
So the target is concrete before any real validation exists: below is a `Report` for a
**fictional** submission with made-up numbers. Nothing here says anything about the seven.
Note the mechanics: every finding is a name, a number, a verdict, and one sentence of
evidence; the section verdict summarises its findings; the overall verdict follows the
aggregation rule (any section FAIL &rArr; FAIL, else any SUSPECT &rArr; SUSPECT); and a
SUSPECT report with an empty `conditions` list is an unfinished report.

In [ ]:
from report import Report, SectionResult, Finding, grid, SECTIONS

example = Report(
    submission="worked-example (not a real submission)",
    sections=[
        SectionResult("luck", "PASS", [
            Finding("bootstrap_sharpe_ci95", (0.8, 2.9), "PASS", "zero outside the interval"),
            Finding("deflated_sharpe", 1.1, "PASS", "at the 12 declared trials"),
        ]),
        SectionResult("lineage", "PASS", [
            Finding("future_info_regression", 0.02, "PASS",
                    "trades do not predict not-yet-published forecast revisions"),
        ]),
        SectionResult("friction", "PASS", [
            Finding("spread_coverage", 2.4, "PASS", "edge per trade = 2.4x half-spread"),
        ]),
        SectionResult("shelf_life", "SUSPECT", [
            Finding("rolling_sharpe_trend", -0.6, "SUSPECT", "edge decaying over the final quarter"),
        ]),
        SectionResult("evidence", "PASS", [
            Finding("min_track_record_days", 140, "PASS", "history 400d exceeds MinTRL 140d"),
        ]),
        SectionResult("warranty", "PASS", [
            Finding("bootstrap_envelope", "fig: envelope.png", "INFO",
                    "live P&L bands + alarm thresholds attached"),
        ]),
    ],
    conditions=["re-validate after 60 live days",
                "kill if 20-day live P&L breaches the bootstrap 5th percentile"],
)
example.verdict

## Where to go from here
The league table above is exactly the evidence a naive desk would fund on. The six
sections in the handout are how you take it apart. A few natural first moves:

- **Friction** — `repricer.py` reprices at mid only, deliberately. The book carries
  three price/size levels per side; build the crossing/walking repricer and re-run the
  league table at prices you'd actually get. One submission will not survive the trip.
- **Luck** — bootstrap and permutation on daily P&L need only the series you just
  built; deflated Sharpe needs only `declared_trials` from the registry.
- **Lineage** — join each blotter against `forecasts.csv` *point-in-time*
  (`issue_ts <= exec_ts`) and ask whether trades know things that weren't knowable yet.
- **Shelf-life** — the curves above already hint that some edges have opinions about
  the calendar.

The contract to build against is `report.py`. The same `run()` must produce all seven
verdicts — no per-submission special cases.